In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models.vision_transformer import VisionTransformer
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from PIL import Image
import os
import matplotlib.pyplot as plt

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

In [3]:
DATA_DIR = './dataset/images'

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [5]:
full_dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

In [6]:
train_size

1626

In [7]:
test_size

407

In [8]:
train_db, test_db = torch.utils.data.random_split(full_dataset, [train_size, test_size])

In [9]:
BATCH_SIZE = 8
train_loader = DataLoader(train_db, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_db, batch_size=BATCH_SIZE, shuffle=False)

In [10]:
class_names = full_dataset.classes # ['airplane', 'face', 'motorcycle']

# model_scratch

In [11]:
NUM_CLASSES = 3

model_scratch = VisionTransformer(
    image_size=224,
    patch_size=16,
    num_layers=12,
    num_heads=12,
    hidden_dim=768,
    mlp_dim=3072,
    num_classes=NUM_CLASSES
)

In [12]:
model_scratch = model_scratch.to(DEVICE)

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_scratch.parameters(), lr=1e-4)

In [14]:
EPOCHS = 3

for epoch in range(EPOCHS):
    model_scratch.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model_scratch(images) 
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Scratch - Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

Scratch - Epoch 1/3, Loss: 0.8316
Scratch - Epoch 2/3, Loss: 0.5116
Scratch - Epoch 3/3, Loss: 0.3395


In [15]:
model_scratch.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model_scratch(images) 
        _, pred = torch.max(outputs, 1)
        correct += (pred == labels).sum().item()

total_acc = 100 * correct / len(test_db)
print(f"model_scratch: {total_acc:.2f}%")

model_scratch: 89.68%


# model_pretrained

In [16]:
model_pretrained = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)

In [17]:
model_pretrained.heads.head = nn.Linear(model_pretrained.heads.head.in_features, NUM_CLASSES)

In [18]:
model_pretrained = model_pretrained.to(DEVICE)

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer_pt = optim.Adam(model_pretrained.parameters(), lr=1e-4)

In [20]:
model_pretrained = model_pretrained.to(DEVICE)

In [21]:
for epoch in range(EPOCHS):
    model_pretrained.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        optimizer_pt.zero_grad() 
        outputs = model_pretrained(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_pt.step()
        total_loss += loss.item()
    
    print(f"Pretrained - Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

Pretrained - Epoch 1/3, Loss: 0.0479
Pretrained - Epoch 2/3, Loss: 0.0004
Pretrained - Epoch 3/3, Loss: 0.0001


In [22]:
model_pretrained.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model_pretrained(images)
        _, pred = torch.max(outputs, 1)
        correct += (pred == labels).sum().item()

total_acc = 100 * correct / len(test_db)
print(f"model_pretrained: {total_acc:.2f}%")

model_pretrained: 100.00%
